# Generacion de Queries Sinteticas para Dataset LTR
## DSRP - Curso de Ingenieria de ML

Este notebook genera queries sinteticas diversas usando:
- **Ollama (Llama 3.2:3b)** para generacion creativa de queries
- **Plantillas basadas en reglas** como queries de linea base

El objetivo es crear un dataset de Learning-to-Rank (LTR) para entrenar modelos de recomendacion.

**Prerequisito**: Ejecutar primero `feature_engineering.ipynb` para generar:
- `data/complete_imdb_database.parquet`
- `data/movie_embs.npy`

In [2]:
import requests
import numpy as np
import polars as pl
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

# Utilidades compartidas
from ml_utils import (
    extract_top_genres,
    extract_decades,
    generate_template_queries,
    normalize_embeddings,
    get_candidates_for_query,
    compute_relevance_score,
    assign_relevance_labels,
    DEFAULT_EMBEDDING_MODEL,
    DEFAULT_TOP_K_CANDIDATES,
    DEFAULT_N_LABEL_BINS,
)

/Users/miguelarquezabdala/repos/dsrp-machine-learning-engineering-4/notebooks/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Configuracion

In [3]:
# Configuracion de Ollama
OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "llama3.2:3b"

# Rutas de datos
DATA_PATH = "data/complete_imdb_database.parquet"
EMBEDDINGS_PATH = "data/movie_embs.npy"
OUTPUT_PATH = "data/ltr_imdb_dataset.parquet"

# Parametros LTR
TOP_K_CANDIDATES = DEFAULT_TOP_K_CANDIDATES
N_LABEL_BINS = DEFAULT_N_LABEL_BINS

## 2. Cargar Datos y Modelos

In [4]:
# Cargar base de datos de peliculas
movies_df = pl.read_parquet(DATA_PATH)
print(f"Se cargaron {movies_df.height} peliculas")
print(f"Columnas: {movies_df.columns}")

# Agregar imdb_votes_log si no existe
if "imdb_votes_log" not in movies_df.columns:
    movies_df = movies_df.with_columns(
        pl.col("imdb_votes").log1p().alias("imdb_votes_log")
    )

movies_df.head(3)

Se cargaron 47203 peliculas
Columnas: ['imdb_id', 'title', 'year', 'genres', 'imdb_rating', 'imdb_votes', 'Runtime', 'Director', 'Actors', 'Plot', 'Country', 'Language']


imdb_id,title,year,genres,imdb_rating,imdb_votes,Runtime,Director,Actors,Plot,Country,Language,imdb_votes_log
str,str,i32,str,f64,i64,str,str,str,str,str,str,f64
"""tt0002423""","""Passion""",1919,"""Biography,Drama,Romance""",6.7,1105,"""113 min""","""Ernst Lubitsch""","""Pola Negri, Emil Jannings, Har…","""The story of Madame DuBarry, t…","""Germany""","""None, German""",7.008505
"""tt0004181""","""Judith of Bethulia""",1914,"""Drama""",6.2,1525,"""61 min""","""D.W. Griffith""","""Blanche Sweet, Henry B. Waltha…","""A fascinating work of high art…","""United States""","""None, English""",7.330405
"""tt0004465""","""The Perils of Pauline""",1914,"""Action,Adventure,Drama""",6.3,1116,"""199 min""","""Louis J. Gasnier, Donald MacKe…","""Pearl White, Crane Wilbur, Pau…","""Young Pauline is left a lot of…","""United States""","""None, English""",7.018402


In [5]:
# Cargar y normalizar embeddings
movie_embs = np.load(EMBEDDINGS_PATH).astype("float32")
movie_embs_norm = normalize_embeddings(movie_embs)
print(f"Dimensiones de embeddings: {movie_embs.shape}")

Dimensiones de embeddings: (47203, 384)


In [6]:
# Cargar modelo de embeddings
model = SentenceTransformer(DEFAULT_EMBEDDING_MODEL)
print(f"Modelo cargado: {DEFAULT_EMBEDDING_MODEL}")

Modelo cargado: sentence-transformers/all-MiniLM-L6-v2


## 3. Extraer Metadata para Generacion de Queries

In [7]:
# Extraer generos y decadas
top_genres = extract_top_genres(movies_df, top_n=15)
decades = extract_decades(movies_df)

print(f"Generos principales: {top_genres}")
print(f"Decadas: {decades}")

Generos principales: ['Drama', 'Comedy', 'Action', 'Romance', 'Crime', 'Thriller', 'Horror', 'Adventure', 'Mystery', 'Fantasy', 'Biography', 'Documentary', 'Sci-Fi', 'Family', 'History']
Decadas: [1900, 1910, 1920, 1930, 1940, 1950, 1960, 1970, 1980, 1990, 2000, 2010, 2020]


In [8]:
# Seleccionar peliculas populares para contexto del LLM
sample_movies = (
    movies_df
    .filter(pl.col("imdb_votes") > 50000)
    .sort("imdb_rating", descending=True)
    .head(50)
    .select(["title", "genres", "year"])
)

sample_titles = sample_movies["title"].to_list()[:20]
print(f"Peliculas populares de muestra: {sample_titles[:5]}")

Peliculas populares de muestra: ['The Shawshank Redemption', 'The Godfather', 'The Dark Knight', '12 Angry Men', 'The Godfather Part II']


## 4. Generacion de Queries con Ollama

Usar Llama 3.2 para generar queries de busqueda de peliculas diversas y en lenguaje natural.

In [9]:
def query_ollama(prompt: str, model: str = OLLAMA_MODEL) -> str:
    """Consultar API de Ollama y retornar texto generado."""
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0.8,
            "top_p": 0.9,
            "num_predict": 200,
        }
    }
    
    try:
        response = requests.post(OLLAMA_URL, json=payload, timeout=30)
        response.raise_for_status()
        return response.json().get("response", "")
    except requests.exceptions.RequestException as e:
        print(f"Error de Ollama: {e}")
        return ""


def parse_queries_from_response(response: str) -> list[str]:
    """Parsear lista numerada de queries de la respuesta del LLM."""
    queries = []
    for line in response.strip().split("\n"):
        line = line.strip()
        if line and line[0].isdigit():
            parts = line.split(".", 1) if "." in line[:3] else line.split(")", 1)
            if len(parts) > 1:
                line = parts[1].strip()
        elif line.startswith("- "):
            line = line[2:].strip()
        
        line = line.strip("\"'")
        if line and len(line) > 5 and len(line) < 100:
            queries.append(line.lower())
    
    return queries

In [10]:
# Probar conexion con Ollama
test_response = query_ollama("Say 'Ollama is working' in one line.")
ollama_available = bool(test_response)
print(f"Ollama disponible: {ollama_available}")
if test_response:
    print(f"Respuesta: {test_response}")

Ollama disponible: True
Respuesta: Ollama is working.


In [11]:
def generate_llm_queries(category: str, context: str, num_queries: int = 5) -> list[dict]:
    """Generar queries diversas usando LLM para una categoria especifica."""
    
    prompt = f"""You are helping create a movie search dataset. Generate {num_queries} diverse, natural movie search queries for: {category}

Context: {context}

Requirements:
- Each query should be how a real user would search for movies
- Vary the phrasing (some formal, some casual)
- Include different intents: browsing, specific mood, recommendations
- Keep queries between 3-12 words
- Output ONLY the queries as a numbered list, nothing else

Generate {num_queries} queries:"""

    response = query_ollama(prompt)
    queries = parse_queries_from_response(response)
    
    return [
        {
            "query_text": q,
            "intent_type": "llm_generated",
            "category": category,
            "emphasis": "neutral",
            "genre": None,
            "decade": None,
        }
        for q in queries[:num_queries]
    ]

In [12]:
llm_queries = []

if ollama_available:
    # Queries basadas en genero
    print("Generando queries basadas en genero...")
    for genre in tqdm(top_genres[:8]):
        queries = generate_llm_queries(
            category=f"{genre} movies",
            context=f"Genre: {genre}. Popular examples exist in our database.",
            num_queries=5
        )
        for q in queries:
            q["genre"] = genre
        llm_queries.extend(queries)
    
    # Queries basadas en estado de animo
    print("Generando queries basadas en estado de animo...")
    moods = [
        ("relaxing weekend", "Movies for a relaxing weekend"),
        ("date night", "Romantic or engaging movies for couples"),
        ("family movie night", "Family-friendly movies"),
        ("mind-bending", "Complex, thought-provoking films"),
        ("adrenaline rush", "Action-packed, exciting movies"),
        ("hidden gems", "Underrated quality films"),
    ]
    
    for mood, context in tqdm(moods):
        queries = generate_llm_queries(category=mood, context=context, num_queries=5)
        llm_queries.extend(queries)
    
    # Queries de similitud
    print("Generando queries de similitud...")
    for title in tqdm(sample_titles[:8]):
        queries = generate_llm_queries(
            category=f"movies similar to {title}",
            context=f"Reference movie: {title}",
            num_queries=3
        )
        for q in queries:
            q["intent_type"] = "similarity_search"
        llm_queries.extend(queries)

print(f"Total queries LLM: {len(llm_queries)}")

Generando queries basadas en genero...


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 8/8 [00:19<00:00,  2.48s/it]


Generando queries basadas en estado de animo...


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6/6 [00:14<00:00,  2.50s/it]


Generando queries de similitud...


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 8/8 [00:13<00:00,  1.70s/it]

Total queries LLM: 94


In [25]:
llm_queries[0]

{'query_text': 'what are some good emotional dramas to watch this weekend?',
 'intent_type': 'llm_generated',
 'category': 'Drama movies',
 'emphasis': 'neutral',
 'genre': 'Drama',
 'decade': None,
 'query_id': 1}

## 5. Queries Basadas en Plantillas

Generar queries estructuradas usando plantillas para cobertura sistematica.

In [16]:
# Generar queries usando plantillas
template_queries = generate_template_queries(top_genres, decades)
print(f"Se generaron {len(template_queries)} queries de plantilla")

Se generaron 280 queries de plantilla


## 6. Combinar y Deduplicar Queries

In [20]:
# Combinar todas las queries
all_queries = llm_queries + template_queries

# Deduplicar por query_text
seen = set()
unique_queries = []
for q in all_queries:
    text = q["query_text"].lower().strip()
    if text not in seen:
        seen.add(text)
        unique_queries.append(q)

# Asignar IDs de query
for i, q in enumerate(unique_queries, start=1):
    q["query_id"] = i

# Crear DataFrame
queries_df = pl.DataFrame(
    unique_queries,
    schema={
        "query_text": pl.Utf8,
        "intent_type": pl.Utf8,
        "category": pl.Utf8,
        "emphasis": pl.Utf8,
        "genre": pl.Utf8,
        "decade": pl.Int64,
        "query_id": pl.Int64,
    }
)

print(f"Total queries unicas: {queries_df.height}")
queries_df.head(10)

Total queries unicas: 374


query_text,intent_type,category,emphasis,genre,decade,query_id
str,str,str,str,str,i64,i64
"""what are some good emotional d…","""llm_generated""","""Drama movies""","""neutral""","""Drama""",null,1
"""looking for a movie about over…","""llm_generated""","""Drama movies""","""neutral""","""Drama""",null,2
"""can you recommend classic dram…","""llm_generated""","""Drama movies""","""neutral""","""Drama""",null,3
"""i'm in the mood for something …","""llm_generated""","""Drama movies""","""neutral""","""Drama""",null,4
"""any highly rated films featuri…","""llm_generated""","""Drama movies""","""neutral""","""Drama""",null,5
"""funny movies for girls""","""llm_generated""","""Comedy movies""","""neutral""","""Comedy""",null,6
"""best comedies of all time""","""llm_generated""","""Comedy movies""","""neutral""","""Comedy""",null,7
"""movies like superbad and pinea…","""llm_generated""","""Comedy movies""","""neutral""","""Comedy""",null,8
"""relaxing comedy movies on netf…","""llm_generated""","""Comedy movies""","""neutral""","""Comedy""",null,9


In [21]:
# Distribucion de queries
print("\nQueries por tipo de intencion:")
print(queries_df.group_by("intent_type").len().sort("len", descending=True))

print("\nQueries por enfasis:")
print(queries_df.group_by("emphasis").len().sort("len", descending=True))


Queries por tipo de intencion:
shape: (8, 2)
┌───────────────────┬─────┐
│ intent_type       ┆ len │
│ ---               ┆ --- │
│ str               ┆ u32 │
╞═══════════════════╪═════╡
│ genre_decade      ┆ 150 │
│ genre_only        ┆ 90  │
│ llm_generated     ┆ 70  │
│ similarity_search ┆ 24  │
│ mood_dark         ┆ 10  │
│ mood_family       ┆ 10  │
│ mood_intense      ┆ 10  │
│ mood_feel_good    ┆ 10  │
└───────────────────┴─────┘

Queries por enfasis:
shape: (3, 2)
┌────────────┬─────┐
│ emphasis   ┆ len │
│ ---        ┆ --- │
│ str        ┆ u32 │
╞════════════╪═════╡
│ neutral    ┆ 199 │
│ rating     ┆ 110 │
│ popularity ┆ 65  │
└────────────┴─────┘


## 7. Recuperacion de Candidatos

Para cada query, recuperar las top-K peliculas candidatas usando similitud de embeddings.

In [22]:
print(f"Recuperando top-{TOP_K_CANDIDATES} candidatos para {queries_df.height} queries...")

all_candidates = []
for q_row in tqdm(queries_df.iter_rows(named=True), total=queries_df.height):
    cand = get_candidates_for_query(
        query_id=q_row["query_id"],
        query_text=q_row["query_text"],
        movies_df=movies_df,
        movie_embs_norm=movie_embs_norm,
        model=model,
        k=TOP_K_CANDIDATES,
    )
    all_candidates.append(cand)

candidates_df = pl.concat(all_candidates)
print(f"\nTotal candidatos: {candidates_df.height}")
candidates_df.head(5)

Recuperando top-100 candidatos para 374 queries...


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 374/374 [00:09<00:00, 38.76it/s]


Total candidatos: 37400


query_id,query_text,imdb_id,title,sim_embedding,imdb_rating,imdb_votes_log,year,genres
i32,str,str,str,f32,f64,f64,i32,str
1,"""what are some good emotional d…","""tt13049760""","""The Last Matinee""",0.522788,5.8,7.948032,2020,"""Horror,Thriller"""
1,"""what are some good emotional d…","""tt0040798""","""Sleep, My Love""",0.508304,6.8,7.803027,1948,"""Drama,Film-Noir,Mystery"""
1,"""what are some good emotional d…","""tt6923740""","""Stella's Last Weekend""",0.501445,6.4,7.014814,2018,"""Comedy,Drama"""
1,"""what are some good emotional d…","""tt15509506""","""The 100""",0.499419,7.4,8.66768,2025,"""Action"""
1,"""what are some good emotional d…","""tt0100998""","""Dreams""",0.497237,7.7,10.370048,1990,"""Drama,Fantasy"""


## 8. Scoring de Relevancia y Etiquetado

Calcular scores de relevancia basados en:
- Similitud de embeddings
- Rating de IMDB
- Popularidad (votos)

Los pesos varian segun el enfasis de la query.

In [23]:
print("Calculando scores de relevancia y etiquetas...")

queries_by_id = {row["query_id"]: row for row in queries_df.iter_rows(named=True)}
ltr_chunks = []

for qid, q_row in tqdm(queries_by_id.items(), total=len(queries_by_id)):
    cand = candidates_df.filter(pl.col("query_id") == qid)
    if cand.is_empty():
        continue

    # Agregar score de relevancia
    cand = compute_relevance_score(cand, emphasis=q_row.get("emphasis", "neutral"))

    # Agregar etiquetas discretas
    cand = assign_relevance_labels(cand, n_bins=N_LABEL_BINS)

    ltr_chunks.append(cand)

ltr_df = pl.concat(ltr_chunks)
print(f"\nTamano del dataset LTR: {ltr_df.height}")

Calculando scores de relevancia y etiquetas...


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 374/374 [00:00<00:00, 528.54it/s]


Tamano del dataset LTR: 37400


## 9. Dataset Final

In [24]:
print(f"Forma del dataset LTR final: {ltr_df.shape}")
print(f"\nColumnas: {ltr_df.columns}")
ltr_df.head(20)

Forma del dataset LTR final: (37400, 11)

Columnas: ['query_id', 'query_text', 'imdb_id', 'title', 'sim_embedding', 'imdb_rating', 'imdb_votes_log', 'year', 'genres', 'rel_score', 'label']


query_id,query_text,imdb_id,title,sim_embedding,imdb_rating,imdb_votes_log,year,genres,rel_score,label
i32,str,str,str,f32,f64,f64,i32,str,f64,i32
1,"""what are some good emotional d…","""tt0175880""","""Magnolia""",0.441414,8.0,12.752733,1999,"""Drama""",0.666602,4
1,"""what are some good emotional d…","""tt5867800""","""Aruvi""",0.482757,8.3,9.614672,2016,"""Drama""",0.653299,4
1,"""what are some good emotional d…","""tt0100998""","""Dreams""",0.497237,7.7,10.370048,1990,"""Drama,Fantasy""",0.645162,4
1,"""what are some good emotional d…","""tt1714210""","""Weekend""",0.48599,7.6,10.440186,2011,"""Drama,Romance""",0.637598,4
1,"""what are some good emotional d…","""tt0102536""","""Night on Earth""",0.425935,7.7,11.156865,1991,"""Comedy,Drama""",0.627132,4
…,…,…,…,…,…,…,…,…,…,…
1,"""what are some good emotional d…","""tt0040798""","""Sleep, My Love""",0.508304,6.8,7.803027,1948,"""Drama,Film-Noir,Mystery""",0.579362,4
1,"""what are some good emotional d…","""tt0062480""","""Weekend""",0.434616,6.9,9.712327,1967,"""Adventure,Comedy,Drama""",0.579344,4
1,"""what are some good emotional d…","""tt8522006""","""Happiest Season""",0.416737,6.6,10.853909,2020,"""Comedy,Romance""",0.575414,4


In [26]:
# Distribucion de etiquetas
print("\nDistribucion de etiquetas:")
print(ltr_df.group_by("label").len().sort("label"))


Distribucion de etiquetas:
shape: (5, 2)
┌───────┬──────┐
│ label ┆ len  │
│ ---   ┆ ---  │
│ i32   ┆ u32  │
╞═══════╪══════╡
│ 0     ┆ 7480 │
│ 1     ┆ 7480 │
│ 2     ┆ 7480 │
│ 3     ┆ 7480 │
│ 4     ┆ 7480 │
└───────┴──────┘


In [27]:
# Muestra de una query
sample_qid = ltr_df["query_id"].unique()[0]
sample_query = ltr_df.filter(pl.col("query_id") == sample_qid)

print(f"Query de muestra: '{sample_query['query_text'][0]}'")
print(f"\nTop 10 resultados:")
sample_query.head(10)

Query de muestra: 'what are some good emotional dramas to watch this weekend?'

Top 10 resultados:


query_id,query_text,imdb_id,title,sim_embedding,imdb_rating,imdb_votes_log,year,genres,rel_score,label
i32,str,str,str,f32,f64,f64,i32,str,f64,i32
1,"""what are some good emotional d…","""tt0175880""","""Magnolia""",0.441414,8.0,12.752733,1999,"""Drama""",0.666602,4
1,"""what are some good emotional d…","""tt5867800""","""Aruvi""",0.482757,8.3,9.614672,2016,"""Drama""",0.653299,4
1,"""what are some good emotional d…","""tt0100998""","""Dreams""",0.497237,7.7,10.370048,1990,"""Drama,Fantasy""",0.645162,4
1,"""what are some good emotional d…","""tt1714210""","""Weekend""",0.48599,7.6,10.440186,2011,"""Drama,Romance""",0.637598,4
1,"""what are some good emotional d…","""tt0102536""","""Night on Earth""",0.425935,7.7,11.156865,1991,"""Comedy,Drama""",0.627132,4
1,"""what are some good emotional d…","""tt15509506""","""The 100""",0.499419,7.4,8.66768,2025,"""Action""",0.611337,4
1,"""what are some good emotional d…","""tt27078110""","""Just a Minute""",0.447346,8.1,7.618742,2024,"""Comedy""",0.604521,4
1,"""what are some good emotional d…","""tt0061814""","""The Incident""",0.442899,7.6,8.59193,1967,"""Crime,Drama,Thriller""",0.595719,4
1,"""what are some good emotional d…","""tt0061138""","""A Man and a Woman""",0.415467,7.5,9.488351,1966,"""Drama,Romance""",0.592698,4


In [28]:
# Guardar dataset final
ltr_df.write_parquet(OUTPUT_PATH)
print(f"Dataset LTR guardado en {OUTPUT_PATH}")

Dataset LTR guardado en data/ltr_imdb_dataset.parquet


## 10. Resumen

In [29]:
print("=" * 50)
print("RESUMEN DEL DATASET LTR")
print("=" * 50)
print(f"Total de queries: {ltr_df['query_id'].n_unique()}")
print(f"Total de pares query-documento: {ltr_df.height}")
print(f"Candidatos por query: {TOP_K_CANDIDATES}")
print(f"Bins de etiquetas: {N_LABEL_BINS} (0-{N_LABEL_BINS-1})")
print(f"\nArchivo generado: {OUTPUT_PATH}")
print(f"\nSiguiente paso: Ejecutar modeling.ipynb para entrenar el modelo LTR")

RESUMEN DEL DATASET LTR
Total de queries: 374
Total de pares query-documento: 37400
Candidatos por query: 100
Bins de etiquetas: 5 (0-4)

Archivo generado: data/ltr_imdb_dataset.parquet

Siguiente paso: Ejecutar modeling.ipynb para entrenar el modelo LTR
